In [ ]:
!pip install sentence-transformers faiss-cpu

In [1]:
import os
import re
import getpass
import requests
import openai
from openai import OpenAI
import jsonschema
import os
import json
from typing import List, Optional, Dict, Any

import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import faiss

In [2]:
load_dotenv()

# ==== НАСТРОЙКИ ====
OPENAI_API_KEY = getpass.getpass("Введи ваш VseGPT ключ API:")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://api.vsegpt.ru/v1"
OPENAI_MODEL = "openai/gpt-4o-mini"

# Пути к файлам
CSV_CRITERIA_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\Критерии.csv"  # файл во вложении
DOCX_SPEC_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\1_Приложение №1_ТЗ_сейсмика.docx"
OUTPUT_REPORT_DOCX = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\Analysis_Report_SGR.docx"

if not OPENAI_API_KEY or OPENAI_API_KEY == "YOUR_OPENAI_API_KEY_HERE":
    raise ValueError("Не задан OPENAI_API_KEY. Установите переменную окружения или впишите ключ.")

print("✅ Конфигурация загружена")

✅ Конфигурация загружена


In [8]:
class GeneratedCriteria(BaseModel):
    selected_ids: List[int] = Field(..., description="ID критериев из справочника, которые релевантны ТЗ")
    new_criteria: List[Dict[str, Any]] = Field(..., description="Список новых критериев: [{'name': '...', 'description': '...', 'importance': int}]")

class CriterionReasoning(BaseModel):
    """Пошаговое рассуждение для одного критерия (Schema-Guided Reasoning)."""
    
    criterion_id: int = Field(..., description="Порядковый номер критерия")
    criterion_name: str = Field(..., description="Название критерия")
    criterion_description: str = Field(..., description="Описание из CSV")
    
    importance_level: int = Field(..., description="Определи уровень важности этого критерия для данного ТЗ")

    criterion_understanding: str = Field(
        ...,
        description="Объясни, ЧТО проверяет этот критерий (1–2 предложения)."
    )

    relevant_sections: str = Field(
        ...,
        description="Какие разделы/приложения ТЗ релевантны для этого критерия?"
    )

    reasoning_steps: str = Field(
        ...,
        description="Пошагово объясни: Критерий требует X → В документе найдено Y → Вывод Z."
    )

    status: str = Field(
        ...,
        description="Статус выполнения критерия.",
        json_schema_extra={"enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]} # Исправили warning заодно
    )

    quote_or_evidence: str = Field(
        ...,
        description="Прямая цитата из ТЗ или обоснование статуса."
    )
    
    recommendation: Optional[str] = Field(
        default=None,
        description="Рекомендация по улучшению. Если нет — передать null."
    )


class SpecificationAnalysisWithReasoning(BaseModel):
    """Анализ ТЗ с явным SGR для каждого критерия."""
    
    reasoning_schema_used: bool = Field(
        ..., # Убрали default=True, модель должна сама решить или вы жестко задаете это в промпте
        description="Флаг: используется ли Schema-Guided Reasoning."
    )

    overall_summary: str = Field(
        ...,
        description="Общая оценка качества ТЗ (1-2 абзаца)."
    )

    criteria_analysis: List[CriterionReasoning] = Field(
        ...,
        description="Список проверок ТОЛЬКО по критериям, отобранным как релевантные для данного ТЗ."
    )

    additional_criteria: List[CriterionReasoning] = Field(
        default=[],  # Может быть пустым списком, если модель не нашла дополнительных
        description="Дополнительные критерии, выявленные моделью при анализе ТЗ (проблемы, не покрытые базовым списком)."
    )
    
    # Лучше временно убрать Dict[str, Any] или заменить на конкретную модель, 
    # так как 'Any' плохо работает со строгим режимом.
    # Если метрики не критичны, можно закомментировать поле metrics:
    # metrics: Dict[str, str] = Field(..., description="Метрики анализа (ключ-значение).")


In [3]:
def load_criteria_from_csv(csv_path: str) -> List[Dict[str, Any]]:
    """
    Читает критерии из CSV с разделителем ';' (точка с запятой).
    Ожидаемые колонки: Название, Описание, Важность
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV с критериями не найден: {csv_path}")
    
    # Важно: использовать sep=';' и encoding зависит от системы
    df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    
    print(f"Загруженные колонки: {df.columns.tolist()}")
    print(f"Первая строка: {df.iloc[0].to_dict()}")
    
    # Проверка обязательных колонок
    required_cols = ["Название", "Описание", "Важность"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"В CSV отсутствует обязательная колонка '{col}'. Колонки: {df.columns.tolist()}")
    
    criteria = []
    for idx, row in df.iterrows():
        criteria.append({
            "id": idx + 1,  # Порядковый номер
            "name": str(row["Название"]).strip(),
            "description": str(row["Описание"]).strip(),
            "importance": int(row["Важность"]),
        })
    
    print(f"✅ Загружено критериев: {len(criteria)}")
    return criteria


def extract_text_from_docx(docx_path: str) -> str:
    """
    Извлекает текст из DOCX файла.
    """
    if not os.path.exists(docx_path):
        raise FileNotFoundError(f"DOCX файл не найден: {docx_path}")
    
    doc = Document(docx_path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    full_text = "\n".join(paragraphs)
    
    print(f"✅ Текст ТЗ извлечён ({len(full_text)} символов)")
    return full_text

def chunk_text(text: str, chunk_size: int = 800, overlap: int = 200) -> List[str]:
    """
    Простое разбиение текста на перекрывающиеся чанки по символам.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end]
        chunks.append(chunk)
        if end == n:
            break
        start = end - overlap
    return chunks


class SpecRAGIndex:
    """
    Индекс для семантического поиска по ТЗ: эмбеддинги чанков + FAISS.
    """
    def __init__(self, model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.chunks: List[str] = []

    def build(self, text: str, batch_size: int = 32):
        self.chunks = chunk_text(text)
        if not self.chunks:
            raise ValueError("Пустой текст ТЗ – нечего индексировать")

        # Эмбеддинги чанков
        embeddings = self.model.encode(self.chunks, batch_size=batch_size, show_progress_bar=True)
        embeddings = embeddings.astype("float32")

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)  # косинус через нормализацию
        # Нормализация для косинуса
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)

    def search(self, query: str, top_k: int = 5) -> List[str]:
        """
        Возвращает top_k наиболее релевантных чанков для запроса.
        """
        if self.index is None:
            raise RuntimeError("RAG индекс ещё не построен. Вызови build(text).")

        q_emb = self.model.encode([query]).astype("float32")
        faiss.normalize_L2(q_emb)
        scores, idxs = self.index.search(q_emb, top_k)

        idxs = idxs[0]
        result_chunks = []
        for i in idxs:
            if i == -1:
                continue
            result_chunks.append(self.chunks[i])
        return result_chunks

In [5]:
def create_criteria_selection_prompt(spec_text: str, all_criteria: List[Dict[str, Any]]) -> str:
    """
    Промпт для ЭТАПА 1: Анализ ТЗ и формирование списка проверок.
    """
    criteria_list_str = "\n".join([
        f"{c['id']}. {c['name']} (Важность: {c['importance']})\n   {c['description']}"
        for c in all_criteria
    ])
    
    return f"""
    Ты — старший методолог по системному анализу.
    Твоя задача — изучить ТЗ и составить ИДЕАЛЬНЫЙ план проверки (список критериев).
    
    Входные данные:
    1. Текст ТЗ (ниже).
    2. Справочник типовых критериев (ниже).

    Алгоритм работы:
    1. Проанализируй ТЗ: определи тип системы, предметную область и ключевые риски.
    2. Выбери из Справочника ID тех критериев, которые строго применимы к этому ТЗ.
    3. ПРИДУМАЙ (СГЕНЕРИРУЙ) новые критерии [additional_criteria], которых нет в справочнике, но они критичны для этого конкретного ТЗ (специфика предметной области, интеграции, безопасность, НТД и т.д.).
    
    Твоя цель: вернуть JSON со списком ID из справочника и списком новых объектов критериев.
    
    СПРАВОЧНИК КРИТЕРИЕВ:
    {criteria_list_str}
    
    ТЕХНИЧЕСКОЕ ЗАДАНИЕ (начало):
    {spec_text[:6000]}...
    """

def create_criterion_analysis_prompt(
    criterion: Dict[str, Any],
    retrieved_context: str
) -> str:
    """
    Промпт для анализа ОДНОГО критерия по релевантным фрагментам ТЗ.
    """
    return f"""
Ты — ведущий системный аналитик и аудитор технической документации.
Проведи аудит ТЗ по ОДНОМУ критерию, используя ТОЛЬКО предоставленный контекст.

Критерий:
ID: {criterion['id']}
Название: {criterion['name']}
Описание: {criterion['description']}

Контекст (фрагменты ТЗ, извлечённые через семантический поиск):
\"\"\" 
{retrieved_context}
\"\"\"

Инструкция:
1. Понимание: что именно проверяет этот критерий.
2. Найди релевантные фрагменты в контексте.
3. Определи статус: ✅ Выполнено, ⚠️ Частично, или ❌ Не выполнено.
4. Приведи цитаты/обоснования.
5. Дай рекомендацию (если требуется).

Отвечай строго в формате JSON по схеме SpecificationAnalysisWithReasoning.criteria_analysis[0],
но БЕЗ массива — только объект одного CriterionReasoning.
"""


In [6]:
class SpecAnalyzerWithSGR:
    """Анализатор ТЗ с двухэтапным процессом: Генерация критериев -> SGR Анализ."""
    
    def __init__(self, api_key: str, model: str = "openai/gpt-4o-mini"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.vsegpt.ru/v1"
        )
        self.model = model
        # --- RAG ---
        self.rag_index: Optional[SpecRAGIndex] = None
        self.original_spec_text: Optional[str] = None
        print(f"✅ Инициализирован анализатор VseGPT ({model})")
        
    def build_rag_index(self, spec_text: str):
        """
        Строим RAG-индекс по тексту ТЗ (вызывать один раз перед анализом).
        """
        self.original_spec_text = spec_text
        self.rag_index = SpecRAGIndex()
        print("📚 Построение RAG-индекса по ТЗ...")
        self.rag_index.build(spec_text)
        print(f"✅ Индекс построен по {len(self.rag_index.chunks)} чанкам")

    def select_and_generate_criteria(self, spec_text: str, all_criteria: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """ЭТАП 1: Выбирает существующие и генерирует новые критерии."""
        
        prompt = create_criteria_selection_prompt(spec_text, all_criteria)
        
        # Схема для этапа генерации
        generation_schema = {
            "name": "CriteriaSelection",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "selected_ids": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "description": "Список ID критериев из справочника, которые релевантны."
                    },
                    "generated_criteria": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "description": {"type": "string"},
                                "importance": {"type": "integer", "description": "Важность 1-5"}
                            },
                            "required": ["name", "description", "importance"],
                            "additionalProperties": False
                        },
                        "description": "Список НОВЫХ критериев, специфичных для этого ТЗ."
                    }
                },
                "required": ["selected_ids", "generated_criteria"],
                "additionalProperties": False
            }
        }

        print(f"🕵️ ЭТАП 1: Формирование списка критериев...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_schema", "json_schema": generation_schema}
            )
            
            data = json.loads(response.choices[0].message.content)
            
            # Сборка финального списка
            final_list = []
            
            # 1. Добавляем выбранные из CSV
            selected_ids = set(data.get('selected_ids', []))
            for crit in all_criteria:
                if crit['id'] in selected_ids:
                    final_list.append(crit)
            
            # 2. Добавляем сгенерированные (присваиваем новые ID)
            # Начинаем нумерацию после последнего ID из CSV
            max_id = max([c['id'] for c in all_criteria], default=0)
            current_id = max_id + 1
            
            for new_crit in data.get('generated_criteria', []):
                new_crit_obj = {
                    "id": current_id,
                    "name": f"[AI] {new_crit['name']}", # Пометка, что критерий от ИИ
                    "description": new_crit['description'],
                    "importance": new_crit['importance']
                }
                final_list.append(new_crit_obj)
                current_id += 1
                
            print(f"   ✅ Выбрано из базы: {len(selected_ids)}")
            print(f"   ✨ Сгенерировано новых: {len(data.get('generated_criteria', []))}")
            print(f"   📋 Итого к проверке: {len(final_list)}")
            
            return final_list

        except Exception as e:
            print(f"❌ Ошибка на этапе генерации: {e}")
            raise

    def analyze_with_sgr_rag(
        self,
        criteria: List[Dict[str, Any]],
        top_k_chunks: int = 5,
    ) -> SpecificationAnalysisWithReasoning:
        """
        ЭТАП 2 (вариант): Анализ по готовому списку критериев с RAG.
        Для каждого критерия:
          - формируем запрос (name+description),
          - достаём top_k релевантных чанков из RAG-индекса,
          - прогоняем через модель с JSON-схемой одного CriterionReasoning,
          - собираем всё в общий SpecificationAnalysisWithReasoning.
        """
        if self.rag_index is None or self.original_spec_text is None:
            raise RuntimeError(
                "RAG-индекс не инициализирован. Сначала вызови build_rag_index(spec_text)."
            )

        print(f"🔬 ЭТАП 2 (RAG): Детальный анализ ({len(criteria)} критериев)...")

        # Схема только для одного CriterionReasoning
        criterion_schema = {
            "name": "CriterionReasoning",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "criterion_id": {"type": "integer"},
                    "criterion_name": {"type": "string"},
                    "criterion_description": {"type": "string"},
                    "importance_level": {"type": "integer"},
                    "criterion_understanding": {"type": "string"},
                    "relevant_sections": {"type": "string"},
                    "reasoning_steps": {"type": "string"},
                    "status": {
                        "type": "string",
                        "enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"],
                    },
                    "quote_or_evidence": {"type": "string"},
                    "recommendation": {"type": ["string", "null"]},
                },
                "required": [
                    "criterion_id",
                    "criterion_name",
                    "criterion_description",
                    "importance_level",
                    "criterion_understanding",
                    "relevant_sections",
                    "reasoning_steps",
                    "status",
                    "quote_or_evidence",
                    "recommendation",
                ],
                "additionalProperties": False,
            },
        }

        criteria_results: List[Dict[str, Any]] = []

        for crit in criteria:
            query = f"{crit['name']}. {crit['description']}"
            # Ищем релевантные чанки
            chunks = self.rag_index.search(query=query, top_k=top_k_chunks)
            retrieved_context = "\n\n---\n\n".join(chunks)

            prompt = create_criterion_analysis_prompt(
                criterion=crit,
                retrieved_context=retrieved_context,
            )

            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_schema", "json_schema": criterion_schema},
            )

            crit_obj = json.loads(response.choices[0].message.content)
            criteria_results.append(crit_obj)

        # Собираем общий объект под твою Pydantic-модель
        overall_summary = (
            "Сводный анализ ТЗ выполнен с использованием RAG-подхода: "
            "для каждого критерия использовались только семантически релевантные фрагменты ТЗ."
        )

        analysis_dict = {
            "reasoning_schema_used": True,
            "overall_summary": overall_summary,
            "criteria_analysis": criteria_results,
            # RAG-версия не добавляет дополнительных критериев на этом этапе:
            "additional_criteria": [],
        }

        return SpecificationAnalysisWithReasoning(**analysis_dict)




In [9]:
def export_sgr_analysis_to_docx(
    analysis: SpecificationAnalysisWithReasoning,
    source_docx_path: str,
    output_path: str
):
    """
    Экспортирует результат SGR-анализа в DOCX-отчёт.
    """
    doc = Document()

    # Заголовок
    title = doc.add_heading("Отчёт по анализу ТЗ", level=1)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph(f"Исходный документ: {os.path.basename(source_docx_path)}")
    doc.add_paragraph(" ")

    # Общая оценка
    doc.add_heading("1. Общая оценка", level=2)
    doc.add_paragraph(analysis.overall_summary)

    # Метрики
    '''if analysis.metrics:
        doc.add_heading("2. Метрики", level=2)
        for k, v in analysis.metrics.items():
            doc.add_paragraph(f"• {k}: {v}")
    doc.add_paragraph(" ")'''

    # Детальный анализ
    doc.add_heading("3. Детальный анализ по критериям", level=2)

    for crit in analysis.criteria_analysis:
        # Заголовок критерия
        heading_text = f"3.{crit.criterion_id} {crit.criterion_name} (Важность: {crit.importance_level})"
        doc.add_heading(heading_text, level=3)

        # Статус (жирный)
        p = doc.add_paragraph()
        run = p.add_run(f"Статус: ")
        run.bold = True
        p.add_run(crit.status)

        # Описание из CSV (серый текст)
        doc.add_paragraph(f"Определение: {crit.criterion_description}", style="List Bullet")

        # Понимание
        '''p = doc.add_paragraph()
        run = p.add_run("Понимание критерия: ")
        run.bold = True
        doc.add_paragraph(crit.criterion_understanding)'''

        # Релевантные разделы
        p = doc.add_paragraph()
        run = p.add_run("Релевантные разделы ТЗ: ")
        run.bold = True
        doc.add_paragraph(crit.relevant_sections)

        # Рассуждение
        '''p = doc.add_paragraph()
        run = p.add_run("Логическое рассуждение: ")
        run.bold = True
        doc.add_paragraph(crit.reasoning_steps)'''

        # Доказательство
        p = doc.add_paragraph()
        run = p.add_run("Доказательство / цитата: ")
        run.bold = True
        doc.add_paragraph(crit.quote_or_evidence)

        # Рекомендация (если есть)
        if crit.recommendation and crit.recommendation.lower() != "нет":
            p = doc.add_paragraph()
            run = p.add_run("Рекомендация: ")
            run.bold = True
            doc.add_paragraph(crit.recommendation)

        
        doc.add_paragraph(" ")  # Разделитель

    # Сохранение
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    doc.save(output_path)
    print(f"✅ Отчёт сохранён: {output_path}")


In [10]:
print("=" * 60)
print("ДВУХЭТАПНЫЙ АНАЛИЗ ТЗ: ГЕНЕРАЦИЯ + ПРОВЕРКА")
print("=" * 60)

# 1. Загружаем критерии из CSV
try:
    base_criteria = load_criteria_from_csv(CSV_CRITERIA_PATH)
except Exception as e:
    print(f"❌ Ошибка при чтении CSV: {e}")
    raise

# 2. Извлекаем текст ТЗ
try:
    spec_text = extract_text_from_docx(DOCX_SPEC_PATH)
except Exception as e:
    print(f"❌ Ошибка при чтении DOCX: {e}")
    raise

# 3. Инициализация
analyzer_sgr = SpecAnalyzerWithSGR(api_key=OPENAI_API_KEY, model=OPENAI_MODEL)

# 3.1. Строим RAG-индекс по ТЗ
analyzer_sgr.build_rag_index(spec_text)

try:
    # --- ШАГ 1: Отбор и Генерация критериев ---
    final_criteria_pool = analyzer_sgr.select_and_generate_criteria(
        spec_text=spec_text,
        all_criteria=base_criteria
    )

    # --- ШАГ 2: Детальный анализ (RAG-версия) ---
    analysis_result = analyzer_sgr.analyze_with_sgr_rag(
        criteria=final_criteria_pool,
        top_k_chunks=5,
    )

    # 4. Вывод результатов в консоль
    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТЫ АНАЛИЗА (Топ-5 критериев)")
    print("=" * 60)

    for crit in analysis_result.criteria_analysis[:5]:
        print(f"\n📌 {crit.criterion_name} (ID: {crit.criterion_id})")
        print(f"   Статус: {crit.status}")
        print(f"   Вывод: {crit.quote_or_evidence[:100]}...")

    # 5. Экспорт
    export_sgr_analysis_to_docx(
        analysis=analysis_result,
        source_docx_path=DOCX_SPEC_PATH,
        output_path=OUTPUT_REPORT_DOCX
    )

except Exception as e:
    print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА ПРОЦЕССА: {e}")
    # Для отладки можно вывести traceback
    import traceback
    traceback.print_exc()

print("\n" + "=" * 60)
print("✅ ГОТОВО! Отчет сформирован.")
print("=" * 60)


ДВУХЭТАПНЫЙ АНАЛИЗ ТЗ: ГЕНЕРАЦИЯ + ПРОВЕРКА
Загруженные колонки: ['Название', 'Описание', 'Важность']
Первая строка: {'Название': 'Полнота функциональных требований', 'Описание': 'Оценка того, насколько подробно описано, что должна делать система, включая объекты, бизнес-логику, роли и отчётность.', 'Важность': 3}
✅ Загружено критериев: 26
✅ Текст ТЗ извлечён (9140 символов)
✅ Инициализирован анализатор VseGPT (openai/gpt-4o-mini)
📚 Построение RAG-индекса по ТЗ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Индекс построен по 15 чанкам
🕵️ ЭТАП 1: Формирование списка критериев...
   ✅ Выбрано из базы: 7
   ✨ Сгенерировано новых: 5
   📋 Итого к проверке: 12
🔬 ЭТАП 2 (RAG): Детальный анализ (12 критериев)...

РЕЗУЛЬТАТЫ АНАЛИЗА (Топ-5 критериев)

📌 Полнота функциональных требований (ID: 1)
   Статус: ⚠️ Частично
   Вывод: "Отчет должен быть обоснован и сопровождаться ссылками на требования действующих НТД." Пункт о необх...

📌 Детализация объектов (ID: 2)
   Статус: ❌ Не выполнено
   Вывод: В предоставленных фрагментах не указаны конкретные сущности, их атрибуты или типы данных. Примеры ис...

📌 Бизнес-логика (ID: 3)
   Статус: ❌ Не выполнено
   Вывод: "Выполнить инженерные изыскания в объёме, необходимом для уточнения исходной и определение расчетной...

📌 Интеграции (ID: 8)
   Статус: ❌ Не выполнено
   Вывод: В представленных фрагментах ТЗ отсутствуют упоминания об интеграциях с другими системами, протоколах...

📌 Безопасность (ID: 14)
   Статус: ❌ Не выполнено
   Вывод: В фрагментах док